# Notebook 03 — Automatic Clustering
## Best Case vs. Worst Case

**Key Insight**: When your queries don't align with natural load order, a cluster key reorganizes partitions so Snowflake can prune 99%+ of the data.

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | `account_id` (matches customer lookup pattern) | No clustering on lookup column |
| **After reclustering** | <1% data scanned | Full table scan |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: The Problem — Customer Lookup Requires Full Scan

The `TRANSACTIONS` table is loaded chronologically (great for date-range queries).  
But **customer lookups by `account_id`** must scan the ENTIRE table because rows for any given account are scattered across all partitions.

In [ ]:
-- WORST CASE: account_id lookup on chronologically-loaded table (full scan)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Check Query Profile** → Notice: partitions scanned ≈ total partitions, ~2+ GB scanned for just 50 rows!

---
## Step 2: Solution — Create Clustered Copy Sorted by `account_id`

We create a copy where data is physically organized by `account_id`.  
This simulates what **Automatic Clustering** achieves in the background after you set a cluster key.

In [ ]:
-- Use XL for creating the 500M-row clustered copy
USE WAREHOUSE HOL_XL;

CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED
    CLUSTER BY (account_id)
AS
SELECT * FROM TRANSACTIONS
ORDER BY account_id;

In [ ]:
-- Switch back to XS for query testing
USE WAREHOUSE HOL_XS;

---
## Step 3: Best Case — Same Query, Clustered Table

In [ ]:
-- BEST CASE: Same query on clustered table (99%+ pruning)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Check Query Profile** → Compare:
- **Original table**: ALL partitions scanned, ~2+ GB read
- **Clustered table**: 1-2 partitions scanned, ~0.015 GB read
- **5x faster, 140x less data scanned!**

---
## Step 4: Side-by-Side Metrics

In [ ]:
-- Compare performance
SELECT
    CASE WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED'
         ELSE 'UNCLUSTERED (full scan)' END AS version,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -30, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_type = 'SELECT'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 5: How It Works — Automatic Clustering

In production, you don't create a separate table. You simply add a cluster key:

In [ ]:
-- In production: just add the cluster key — Snowflake reclusters in the background
-- ALTER TABLE TRANSACTIONS CLUSTER BY (account_id);

-- Estimate the ongoing maintenance cost
SELECT SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS('TRANSACTIONS', '(account_id)');

---
## Step 6: Anti-Pattern — Clustering on Low-Cardinality Column

In [ ]:
-- currency_code has only 3 values — already naturally well-clustered
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(currency_code)');

**Takeaway**: Low-cardinality columns are already well-clustered. Adding a cluster key = wasted credits.

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | High-cardinality column you filter on (`account_id`) | Low-cardinality column (`currency_code`) |
| **Result** | 99%+ pruning, 5x+ faster, 140x less data | Zero improvement, wasted credits |
| **When to use** | Queries don't align with natural load order | NEVER cluster on few-value columns |

**Decision framework**:
1. Identify your most common filter column
2. If queries scan most of the table for that filter → add cluster key
3. Estimate cost before committing (`SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS`)
4. Snowflake reclusters automatically in the background — no maintenance needed

**Next** → [Notebook 04 — Warehouse Optimization](./Notebook_04_Warehouse_Optimization.ipynb)